In [1]:
!pip install torch torchvision transformers ultralytics opencv-python-headless numpy -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.1 MB/s eta 0:00:00


In [8]:
from google.colab import files
uploaded = files.upload()

Saving sample1.mp4 to sample1.mp4


In [3]:
import cv2
import numpy as np
import torch
from transformers import AutoImageProcessor, AutoModelForDepthEstimation
from ultralytics import YOLO
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using: {device}")

processor = AutoImageProcessor.from_pretrained("depth-anything/Depth-Anything-V2-Small-hf")
depth_model = AutoModelForDepthEstimation.from_pretrained("depth-anything/Depth-Anything-V2-Small-hf")
depth_model = depth_model.to(device).eval()

yolo = YOLO("yolov8n.pt")
print("All models loaded")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Using: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

The image processor of type `DPTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json:   0%|          | 0.00/950 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/99.2M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/287 [00:00<?, ?it/s]

All models loaded


In [13]:
def make_bev(depth_raw, boxes_with_risk, bev_size=300):
    bev = np.zeros((bev_size, bev_size, 3), dtype=np.uint8)

    for i in range(0, bev_size, bev_size // 5):
        cv2.line(bev, (0, i), (bev_size, i), (40, 40, 40), 1)
        cv2.line(bev, (i, 0), (i, bev_size), (40, 40, 40), 1)

    cam_x, cam_y = bev_size // 2, bev_size - 20
    cv2.drawMarker(bev, (cam_x, cam_y), (255, 255, 255), cv2.MARKER_TRIANGLE_UP, 15, 2)

    d_min = depth_raw.min()
    d_max = depth_raw.max()

    for (cx, depth_val, color, label) in boxes_with_risk:
        bev_x = int((cx / depth_raw.shape[1]) * bev_size)
        depth_norm = (depth_val - d_min) / (d_max - d_min + 1e-8)
        bev_y = int((1.0 - depth_norm) * (bev_size - 40)) + 10
        cv2.circle(bev, (bev_x, bev_y), 8, color, -1)
        cv2.putText(bev, label[:3], (bev_x + 10, bev_y + 4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.35, color, 1)

    cv2.putText(bev, "BEV MAP", (5, 15), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (180, 180, 180), 1)
    return bev


cap = cv2.VideoCapture("sample1.mp4")
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"Video: {total} frames, {fps:.1f}fps, {w}x{h}")

out = cv2.VideoWriter("output.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w * 2 + h, h))

frame_count = 0
last_depth = None
last_depth_raw = None
close_thresh = None
far_thresh = None

with torch.no_grad():
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_count % 3 == 0:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_img = Image.fromarray(rgb)
            inputs = processor(images=pil_img, return_tensors="pt").to(device)
            outputs = depth_model(**inputs)
            depth_raw = outputs.predicted_depth.squeeze().cpu().numpy()
            last_depth_raw = cv2.resize(depth_raw, (w, h))

            close_thresh = float(np.percentile(last_depth_raw, 70))
            far_thresh   = float(np.percentile(last_depth_raw, 40))

            depth_norm    = (depth_raw - depth_raw.min()) / (depth_raw.max() - depth_raw.min() + 1e-8)
            depth_uint8   = (depth_norm * 255).astype(np.uint8)
            depth_resized = cv2.resize(depth_uint8, (w, h))
            last_depth    = cv2.applyColorMap(depth_resized, cv2.COLORMAP_INFERNO)

        results   = yolo(frame, verbose=False)[0]
        annotated = frame.copy()
        obj_count    = len(results.boxes)
        danger_count = 0
        boxes_with_risk = []

        if last_depth_raw is not None and close_thresh is not None:
            for box in results.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                cls   = int(box.cls[0])
                label = yolo.names[cls]
                conf  = float(box.conf[0])
                cx    = (x1 + x2) // 2

                cx1 = x1 + (x2 - x1) // 4
                cy1 = y1 + (y2 - y1) // 4
                cx2 = x2 - (x2 - x1) // 4
                cy2 = y2 - (y2 - y1) // 4
                region    = last_depth_raw[cy1:cy2, cx1:cx2]
                depth_val = float(np.median(region)) if region.size > 0 else 0.0

                if depth_val > close_thresh:
                    color, risk = (0, 0, 255), "DANGER"
                    danger_count += 1
                elif depth_val > far_thresh:
                    color, risk = (0, 165, 255), "WARNING"
                else:
                    color, risk = (0, 255, 0), "SAFE"

                cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)
                cv2.putText(annotated, f"{label} {risk} {conf:.2f}",
                            (x1, y1 - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

                boxes_with_risk.append((cx, depth_val, color, label))

        # ── HUD ──
        cv2.putText(annotated, f"Objects: {obj_count}  Hazards: {danger_count}",
                    (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

        # ── BEV ──
        bev = make_bev(
            last_depth_raw if last_depth_raw is not None else np.zeros((h, w)),
            boxes_with_risk,
            bev_size=h
        )

        depth_display = last_depth if last_depth is not None else np.zeros_like(frame)
        display = np.hstack([annotated, depth_display, bev])
        out.write(display)

        frame_count += 1
        if frame_count % 50 == 0:
            print(f"Frame {frame_count}/{total}")

cap.release()
out.release()
print("Done.")

Video: 481 frames, 24.0fps, 720x1280
Frame 50/481
Frame 100/481
Frame 150/481
Frame 200/481
Frame 250/481
Frame 300/481
Frame 350/481
Frame 400/481
Frame 450/481
Done.


In [14]:
from google.colab import files
files.download("output.mp4")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
import os
print(os.getcwd())

from google.colab import runtime

/content
